# Model predictions and mechanisms

This notebook is for **model interpretation rather than estimation**. It keeps the main figures in a compact sequence:

1. Hand-to-mouth versus heterogeneous assets.
2. Forward hazard, survival, consumption, and asset paths.
3. Value functions from the backward solution.
4. Standard versus reference-dependent predictions.
5. Multi-type selection / dilution.

A key distinction used throughout:
- `SolveModel()` returns policies conditional on **current assets**.
- `SolveForward()` maps those policies into paths conditional on **initial assets**.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL_DIR = os.environ.get(
    "JOB_SEARCH_MODEL_DIR",
    "/Users/jacobaspnissen/Desktop/Økonomi/Dynamic programming/Termpaper/termpaper_dynprog/Job Search/"
)

if MODEL_DIR not in sys.path:
    sys.path.append(MODEL_DIR)

import Model_components as model

FIGURE_DIR = os.path.join(os.getcwd(), "prediction_figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

## 2. Parameters and institutions

In [ ]:
delta = 0.89
gamma = 0.81
eta_rd = 1.0

k = 359.0
k1 = 300.0
k2 = 150.0
k3 = 90.0
q1 = 0.33
q2 = 0.33

lmbda_rd = 4.9
N = 6

abar = [0, 17955]
n_a = 100
n_c = 100

T1 = 6
T2 = 18
T3 = 24
T = 44

b1_pre, b2_pre, b3 = 222, 222, 114
b1_post, b2_post = 342, 171

welfare = 90
w = 675
R = 0.01

institutions_pre = np.array([
    n_a, n_c, T1, T2, T3, T,
    b1_pre, b2_pre, b3, welfare, w, R
])

institutions_post = np.array([
    n_a, n_c, T1, T2, T3, T,
    b1_post, b2_post, b3, welfare, w, R
])

params_std = np.array([delta, gamma, 0.0, k, 0.0, N])
params_rd = np.array([delta, gamma, eta_rd, k, lmbda_rd, N])
params_2type = np.array([delta, gamma, 0.0, k1, 0.0, N, k2, q1])
params_3type = np.array([delta, gamma, 0.0, k1, 0.0, N, k2, k3, q1, q2])

timevec = (np.arange(T) + 1) * 15
timevec_survival = np.arange(T + 1) * 15

asset_rows = [0, 25, 50, 99]

# 3. Hand-to-mouth versus heterogeneous assets

The HTM model has `A_t=0` and `c_t=y_t` in every period. The asset model instead starts workers at different initial asset levels and follows their optimal asset policy forward.

The first figure compares the standard one-type aggregate hazard under these two consumption environments.

In [ ]:
def aggregate_hazard_from_survival(survival, weights):
    survival_total = weights @ survival
    return (
        survival_total[:-1] - survival_total[1:]
    ) / survival_total[:-1]


# Hand-to-mouth
htm = 1
weights_htm = model.make_weights(htm, abar, n_a)

C_htm_pre, S_htm_pre, surv_htm_pre, A_htm_pre, Agrid_htm, benefits_htm_pre =     model.SolveForward(params_std, institutions_pre, abar, htm)

C_htm_post, S_htm_post, surv_htm_post, A_htm_post, _, benefits_htm_post =     model.SolveForward(params_std, institutions_post, abar, htm)

# Heterogeneous assets
htm = 0
weights_assets = model.make_weights(htm, abar, n_a)

C_assets_pre, S_assets_pre, surv_assets_pre, A_assets_pre, Agrid_assets, benefits_assets_pre =     model.SolveForward(params_std, institutions_pre, abar, htm)

C_assets_post, S_assets_post, surv_assets_post, A_assets_post, _, benefits_assets_post =     model.SolveForward(params_std, institutions_post, abar, htm)

haz_htm_pre = aggregate_hazard_from_survival(surv_htm_pre, weights_htm)
haz_htm_post = aggregate_hazard_from_survival(surv_htm_post, weights_htm)

haz_assets_pre = aggregate_hazard_from_survival(surv_assets_pre, weights_assets)
haz_assets_post = aggregate_hazard_from_survival(surv_assets_post, weights_assets)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.plot(timevec, haz_htm_pre, label="HTM pre-reform")
ax.plot(timevec, haz_assets_pre, label="Assets pre-reform")
ax.plot(timevec, haz_htm_post, linestyle="--", label="HTM post-reform")
ax.plot(timevec, haz_assets_post, linestyle="--", label="Assets post-reform")

for x in [T1*15, T2*15, T3*15]:
    ax.axvline(x, linestyle=":", linewidth=1)

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Hazard rate")
ax.set_title("Aggregate model hazards: HTM versus heterogeneous assets")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "htm_vs_assets_hazards.pdf"), bbox_inches="tight")
plt.show()

# 4. Heterogeneous assets: forward paths

These figures should use `SolveForward()`, because the rows then refer to **initial** asset levels. The same row in `SolveModel()` would instead hold current assets fixed and is not an individual unemployment-spell trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for i in asset_rows:
    ax.plot(
        timevec,
        S_assets_pre[i],
        label=f"Initial assets: ${Agrid_assets[i]:,.0f}"
    )

for x in [T2*15, T3*15]:
    ax.axvline(x, linestyle="--", linewidth=1)

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Hazard rate")
ax.set_title("Pre-reform hazard by initial assets")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "hazard_by_initial_assets_pre.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for i in asset_rows:
    ax.plot(
        timevec_survival,
        surv_assets_pre[i],
        label=f"Initial assets: ${Agrid_assets[i]:,.0f}"
    )

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Probability still unemployed")
ax.set_title("Pre-reform survival by initial assets")
ax.set_xlim(0, 560)
ax.set_ylim(0, 1.02)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "survival_by_initial_assets_pre.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for i in asset_rows:
    ax.plot(
        timevec,
        C_assets_pre[i],
        label=f"Initial assets: ${Agrid_assets[i]:,.0f}"
    )

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Consumption")
ax.set_title("Pre-reform consumption by initial assets")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "consumption_by_initial_assets_pre.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for i in asset_rows:
    ax.plot(
        timevec_survival,
        A_assets_pre[i],
        label=f"Initial assets: ${Agrid_assets[i]:,.0f}"
    )

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Assets")
ax.set_title("Pre-reform forward asset paths")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "asset_paths_pre.pdf"), bbox_inches="tight")
plt.show()

## 4.1 Post-reform forward paths

Post-reform is useful for showing anticipation and smoothing around the 90-day front-loaded benefit reduction.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for i in asset_rows:
    ax.plot(
        timevec,
        S_assets_post[i],
        label=f"Initial assets: ${Agrid_assets[i]:,.0f}"
    )

for x in [T1*15, T2*15, T3*15]:
    ax.axvline(x, linestyle="--", linewidth=1)

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Hazard rate")
ax.set_title("Post-reform hazard by initial assets")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "hazard_by_initial_assets_post.pdf"), bbox_inches="tight")
plt.show()

# 5. Value functions

Value functions are naturally conditional on **current assets**, so here we intentionally use `SolveModel()` rather than `SolveForward()`.

Do not describe these lines as paths for workers who started at a given asset level. They answer a different question: what is the value of employment/unemployment in period `t` conditional on currently holding a particular amount of assets?

In [ ]:
htm = 0

S_policy_pre, V_emp_pre, V_uemp_pre, c_emp_pre, c_uemp_pre, Vss_emp_pre, Vss_uemp_pre, css_emp_pre, css_uemp_pre, survival_policy_pre, benefits_policy_pre =     model.SolveModel(params_std, institutions_pre, abar, htm)

timevec_value = np.arange(T + 1) * 15

value_rows = [0, 50, 99]

fig, ax = plt.subplots(figsize=(7.2, 4.8))

for i in value_rows:
    ax.plot(
        timevec_value,
        V_emp_pre[i],
        linestyle="--",
        label=f"VE, current A=${Agrid_assets[i]:,.0f}"
    )
    ax.plot(
        timevec_value,
        V_uemp_pre[i],
        label=f"VU, current A=${Agrid_assets[i]:,.0f}"
    )

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Value")
ax.set_title("Value functions, pre-reform")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "value_functions_pre.pdf"), bbox_inches="tight")
plt.show()

# 6. Standard versus reference dependence

This comparison isolates the reference-dependent mechanism by holding the remaining starting parameters fixed.

In [ ]:
htm = 0
weights = model.make_weights(htm, abar, n_a)

mom_std = model.simulate_moments(
    params_std, institutions_pre, institutions_post, abar, weights, htm
)
mom_rd = model.simulate_moments(
    params_rd, institutions_pre, institutions_post, abar, weights, htm
)

def split_half(arr):
    mid = len(arr) // 2
    return np.asarray(arr[:mid]), np.asarray(arr[mid:])

std_pre, std_post = split_half(mom_std)
rd_pre, rd_post = split_half(mom_rd)

timevec_mom = np.arange(30, 541, 15)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.plot(timevec_mom, std_pre, label="Standard pre")
ax.plot(timevec_mom, rd_pre, linestyle="--", label="RD pre")
ax.plot(timevec_mom, std_post, label="Standard post")
ax.plot(timevec_mom, rd_post, linestyle="--", label="RD post")

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Aggregate hazard")
ax.set_title("Reference dependence versus standard model")
ax.set_xlim(0, 560)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "standard_vs_rd.pdf"), bbox_inches="tight")
plt.show()

# 7. Heterogeneous search-cost types

The most informative mechanism figure is not only type-specific survival, but the **composition of the remaining unemployed population**. Low-`k` types search more, exit faster, and therefore become less prevalent with unemployment duration.

In [ ]:
def unpack_type_structure(params_vec):
    p = np.asarray(params_vec).ravel()

    if len(p) == 8:
        d, g, e, k1_, lam, N_, k2_, q1_ = p
        return [k1_, k2_], np.array([q1_, 1-q1_]), [d, g, e, lam, N_]

    if len(p) == 10:
        d, g, e, k1_, lam, N_, k2_, k3_, q1_, q2_ = p
        return [k1_, k2_, k3_], np.array([q1_, q2_, 1-q1_-q2_]), [d, g, e, lam, N_]

    raise ValueError("Use a 2-type or 3-type parameter vector.")


def type_diagnostics(params_vec, institutions):
    kvals, shares, common = unpack_type_structure(params_vec)
    d, g, e, lam, N_ = common

    type_survival = []

    for k_j in kvals:
        params_j = np.array([d, g, e, k_j, lam, N_])
        _, _, surv_j, _, _, _ = model.SolveForward(
            params_j, institutions, abar, htm
        )
        type_survival.append(weights @ surv_j)

    type_survival = np.asarray(type_survival)

    mass = shares[:, None] * type_survival
    composition = mass / mass.sum(axis=0, keepdims=True)

    return np.asarray(kvals), shares, type_survival, composition


kvals_3, shares_3, surv_3, composition_3 =     type_diagnostics(params_3type, institutions_pre)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for j, k_j in enumerate(kvals_3):
    ax.plot(
        timevec_survival,
        surv_3[j],
        label=f"Type {j+1}: k={k_j:.0f}"
    )

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Probability still unemployed")
ax.set_title("Type-specific survival")
ax.set_xlim(0, 560)
ax.set_ylim(0, 1.02)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "type_survival.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for j, k_j in enumerate(kvals_3):
    ax.plot(
        timevec_survival,
        composition_3[j],
        label=f"Type {j+1}: k={k_j:.0f}"
    )

ax.set_xlabel("Number of days since UI claim")
ax.set_ylabel("Share among remaining unemployed")
ax.set_title("Selection across latent search-cost types")
ax.set_xlim(0, 560)
ax.set_ylim(0, 1)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "type_composition.pdf"), bbox_inches="tight")
plt.show()

# 8. Outputs worth adding

For the final paper, I would prioritize:

1. **Preferred-model fit against empirical hazards** pre and post reform.
2. **Forward asset + consumption paths** for a few initial asset levels.
3. **Type-survival and type-composition plots** if heterogeneous search-cost types are central to the interpretation.
4. **Search-cost curves** implied by the estimated `k` and `gamma`.
5. **Grid sensitivity** for the preferred model.
6. **Initial-asset-distribution sensitivity**, because the current Beta distribution is an explicit modelling assumption.
7. **Weighted residuals by duration**, to show exactly where RD or type heterogeneity improves fit.
8. **Aggregate survival implied by each estimated model**, alongside the empirical survival series.

## Notes for the report

The current draft still describes the old static-asset mistake and says the forward asset update was not corrected. Once the new `SolveForward()` implementation is final, those passages should be rewritten rather than retained as a model flaw.

Also update any text that says interpolation is used for asset values *outside* the grid. With the explicit upper/lower asset constraints, interpolation should be between grid points; extrapolation outside the solved state space should not be part of the intended algorithm.